# 03 Does the embedding model matter?

This notebook is self-contained. It downloads NFCorpus and the models from the Hugging Face Hub, encodes all text, and calculates all numbers and figures. It does not read `results/`, `cache/`, or a `silent_rag` module. You need only the pip packages in the next cell. On a laptop CPU, the run time is approximately 20 to 40 minutes. The notebook encodes the corpus one time for each model, sequence-length, and prefix configuration (approximately nine). `bge-base` is the slowest.

**Questions**
1. Are all-MiniLM, bge-small, and e5-small better than the baseline (multi-qa-MiniLM)?
2. Is the effect large, or only detectable?
3. Do prefixes, sequence length, or a larger model change the results?

**Method**
- Paired differences for each query, on all labelled NFCorpus queries. Each model uses **its own default `max_seq_length`**. A forced-512 design is a sensitivity row.
- Cluster-bootstrap CIs. The clusters are topic groups, because queries on one topic are not independent.
- Sign-flip permutation tests, with Holm correction over the three pre-specified contrasts.
- TOST equivalence, minimum detectable effect, and sensitivity configurations.

Each inferential section has a **"Why this test"** note. The note gives the estimand, the unit of independence, the assumptions, the rejected alternative, and the Monte Carlo resolution.

**Scope.** This notebook uses only NFCorpus. NFCorpus is the development set of the project. SciFact and FiQA are held-out sets for the final run. This notebook does not load them.

In [ ]:
# On a new environment, remove the "#" from the next line.
# %pip install -q datasets==5.1.0 sentence-transformers==6.1.0 torch==2.14.1 scikit-learn==1.9.1 scipy==1.18.1 pandas==3.0.6 numpy==2.5.4 matplotlib=3.11.2

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # On macOS, this prevents an OpenMP conflict. It has no effect on other systems.
os.environ.setdefault("OMP_NUM_THREADS", "1")

import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy import stats
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 30)

SEED = 0
np.random.seed(SEED)
K = 10
SPLITS = ["train", "validation", "test"]  # The Hub name of the qrels split is "validation", not "dev".
GROUP_K = 50      # Number of topic groups for the cluster bootstrap and the group sign-flip
B = 2000          # Number of bootstrap and permutation draws
DELTA = 0.01      # Equivalence margin in nDCG@10. This value was set before the results were examined.
BASELINE = "multi-qa-MiniLM"
MAX_SEQ = None    # None: each model uses its own default max_seq_length (primary). Forced lengths are sensitivity rows.

NFCORPUS = dict(
    repo="BeIR/nfcorpus", revision="b5026a0e96e8a7ac4f95f482a596389289d46269",
    qrels="BeIR/nfcorpus-qrels", qrels_revision="a451b3b26d3ae1358f259c1a3a4dd61fcea35a65",
)
MODELS = {
    "multi-qa-MiniLM": dict(repo="sentence-transformers/multi-qa-MiniLM-L6-cos-v1", revision="b207367332321f8e44f96e224ef15bc607f4dbf0", query_prefix="", doc_prefix=""),
    "all-MiniLM": dict(repo="sentence-transformers/all-MiniLM-L6-v2", revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41", query_prefix="", doc_prefix=""),
    "bge-small": dict(repo="BAAI/bge-small-en-v1.5", revision="5c38ec7c405ec4b44b94cc5a9bb96e735b38267a", query_prefix="Represent this sentence for searching relevant passages: ", doc_prefix=""),
    "e5-small": dict(repo="intfloat/e5-small-v2", revision="ffb93f3bd4047442299a41ebb6fa998a38507c52", query_prefix="query: ", doc_prefix="passage: "),
}

## Helpers and data

This section defines the functions that load the data, encode the text, do an exact cosine search, and calculate nDCG@10. nDCG@10 uses graded gain and a log2 discount, as in BEIR. The notebook keeps the encoders and embeddings in memory. Configurations that share them use the same copy. For example, `e5-small` with and without the query prefix uses the same document vectors.

**The tie rule is different from the benchmark runner.** This notebook breaks exact score ties by document index (stable sort). `silent_rag.pipeline.Retriever` uses FAISS `IndexFlatIP`, which breaks ties in a different way. The corpus has duplicate documents, so some queries have an exact tie at rank 10/11. Thus, mean nDCG@10 here can be different from the clean baseline of the runner in the fourth decimal. This does not change the model contrasts at the reported precision.

In [ ]:
def doc_text(title, text):
    return f"{(title or '').strip()} {(text or '').strip()}".strip()


def load_nfcorpus():
    c = NFCORPUS
    corpus = load_dataset(c["repo"], "corpus", split="corpus", revision=c["revision"])
    queries = load_dataset(c["repo"], "queries", split="queries", revision=c["revision"])
    qtext = {str(q["_id"]): q["text"] for q in queries}
    split_of, qrels = {}, {}
    for split in SPLITS:
        for row in load_dataset(c["qrels"], split=split, revision=c["qrels_revision"]):
            q = str(row["query-id"])
            if split_of.setdefault(q, split) == split:
                qrels.setdefault(q, {})[str(row["corpus-id"])] = int(row["score"])
    doc_ids = [str(d["_id"]) for d in corpus]
    docs = [doc_text(t, x) for t, x in zip(corpus["title"], corpus["text"])]
    return doc_ids, docs, {q: qtext[q] for q in qrels}, qrels, split_of


def ndcg_at_k(ranked, rels, k=K):
    dcg = sum(rels.get(d, 0) / np.log2(i + 2) for i, d in enumerate(ranked[:k]))
    ideal = sorted(rels.values(), reverse=True)[:k]
    idcg = sum(r / np.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def search(D, Q, k=K):
    S = Q @ D.T
    # Stable full sort: if two scores are equal (the corpus has duplicate documents), the lower document index comes first.
    # Thus, the top-k does not depend on the search routine.
    return np.argsort(-S, axis=1, kind="stable")[:, :k]


doc_ids, docs, queries, qrels, split_of = load_nfcorpus()
qids = [q for q in queries if any(r > 0 for r in qrels[q].values())]   # Queries that have one or more relevant documents
split = np.array([split_of[q] for q in qids])
rel = [{d for d, v in qrels[q].items() if v > 0} for q in qids]
print(len(docs), "docs;", len(qids), "queries;", pd.Series(split).value_counts().to_dict())

## Retrieval for every configuration

**Sequence length.** The primary comparison runs each model at **its own default `max_seq_length`**. The notebook reads this value from the loaded model before it changes a setting. A forced length of 512 tokens is not the shipped setting of all models: `all-MiniLM` has a default of 256. For the two MiniLM models, the other length (256 or 512) is a labelled sensitivity row.

The notebook runs the four main models first. Then it runs these sensitivity configurations:
- bge-small without its query instruction
- e5-small without its prefixes, and e5-small with only the query prefix
- the two MiniLM models at their non-default length
- `bge-base-en-v1.5`

All models, including `bge-base`, use a fixed Hub revision.

In [ ]:
bge_base_rev = "a5beb1e3e68b9ab74eb54cfd186867f64f240e1a"   # Revision found on 2026-10-15 and fixed here. This makes the run reproducible.
MODELS["bge-base"] = dict(repo="BAAI/bge-base-en-v1.5", revision=bge_base_rev, query_prefix=MODELS["bge-small"]["query_prefix"], doc_prefix="")

_enc, _doc_emb, _q_emb = {}, {}, {}
DEFAULT_SEQ = {}   # Shipped max_seq_length of each model, read before a change to the setting


def encoder(label, max_seq=None):
    """Return (encoder, max_seq). If max_seq is None, use the default of the model."""
    spec = MODELS[label]
    if label not in DEFAULT_SEQ:
        e = SentenceTransformer(spec["repo"], revision=spec["revision"], device="cpu")
        DEFAULT_SEQ[label] = e.max_seq_length
        _enc[(label, e.max_seq_length)] = e
    max_seq = DEFAULT_SEQ[label] if max_seq is None else max_seq
    if (label, max_seq) not in _enc:
        e = SentenceTransformer(spec["repo"], revision=spec["revision"], device="cpu")
        e.max_seq_length = max_seq
        _enc[(label, max_seq)] = e
    return _enc[(label, max_seq)], max_seq


def encode(enc, texts, batch_size=32):
    return np.ascontiguousarray(enc.encode(list(texts), batch_size=batch_size, normalize_embeddings=True,
                                           convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)


def run_config(label, max_seq=MAX_SEQ, query_prefix=None, doc_prefix=None):
    spec = MODELS[label]
    qp = spec["query_prefix"] if query_prefix is None else query_prefix
    dp = spec["doc_prefix"] if doc_prefix is None else doc_prefix
    enc, max_seq = encoder(label, max_seq)
    if (label, max_seq, dp) not in _doc_emb:
        _doc_emb[(label, max_seq, dp)] = encode(enc, [dp + t for t in docs])
    if (label, max_seq, qp) not in _q_emb:
        _q_emb[(label, max_seq, qp)] = encode(enc, [qp + queries[q] for q in qids])
    D, Q = _doc_emb[(label, max_seq, dp)], _q_emb[(label, max_seq, qp)]
    top = search(D, Q)
    ids = [[doc_ids[j] for j in row] for row in top]
    ndcg = np.array([ndcg_at_k(r, qrels[q]) for r, q in zip(ids, qids)])
    recall = np.array([len(r & set(i)) / len(r) for r, i in zip(rel, ids)])
    return {"Q": Q, "ids": ids, "ndcg": ndcg, "recall": recall, "max_seq": max_seq}


for label in MODELS:
    encoder(label)
print("default max_seq_length:", DEFAULT_SEQ)

# Non-default length for the two MiniLM models: 256 if the default is 512. Otherwise, 512.
ALT = {m: (256 if DEFAULT_SEQ[m] == 512 else 512) for m in (BASELINE, "all-MiniLM")}
ALT_NAME = {m: f"{m} @{ALT[m]}" for m in ALT}

configs = {
    "multi-qa-MiniLM": dict(label="multi-qa-MiniLM"),
    "all-MiniLM": dict(label="all-MiniLM"),
    "bge-small": dict(label="bge-small"),
    "e5-small": dict(label="e5-small"),
    # Sensitivity configurations
    "bge-small, no query prefix": dict(label="bge-small", query_prefix=""),
    "e5-small, no prefixes": dict(label="e5-small", query_prefix="", doc_prefix=""),
    "e5-small, query prefix only": dict(label="e5-small", doc_prefix=""),
    **{ALT_NAME[m]: dict(label=m, max_seq=ALT[m]) for m in ALT},
    "bge-base": dict(label="bge-base"),
}
runs = {}
for cname, kw in configs.items():
    runs[cname] = run_config(**kw)
    print(cname, "| max_seq", runs[cname]["max_seq"], "| mean nDCG@10", round(float(runs[cname]["ndcg"].mean()), 4), flush=True)
nd = pd.DataFrame({k: r["ndcg"] for k, r in runs.items()})

## Accuracy on the test queries

This section uses only the 323 test queries. It shows the result of a test-only design. It gives nDCG@10 with a 95% bootstrap CI over queries, recall@10, and the paired difference from the baseline with a paired bootstrap CI. This section treats queries as independent. The all-query analysis in the next sections includes the topic dependence.

**Why this test**
- **Estimand:** the mean nDCG@10 of each model on the 323 test queries, and the mean paired difference from the baseline.
- **Unit of independence:** the query. This is optimistic, because test queries share topics and relevant documents. The cluster analysis in the next sections is the primary analysis.
- **Assumptions:** the percentile bootstrap needs an approximately symmetric sample mean. With n = 323 and bounded nDCG, this is true.
- **Why not the usual alternative:** a paired t-interval assumes that the differences for each query are approximately normal. They are zero-inflated and bounded. The bootstrap uses only the sampling distribution of the mean.
- **Resolution / Monte Carlo error:** B = 2,000 resamples. The Monte Carlo error on a 2.5% percentile endpoint is approximately 0.001 nDCG. This is much smaller than the CI width.

In [ ]:
def boot_ci(x, n_boot=B, seed=SEED):
    x = np.asarray(x, float)
    rng = np.random.default_rng(seed)
    b = x[rng.integers(0, len(x), (n_boot, len(x)))].mean(axis=1)
    return float(x.mean()), *np.percentile(b, [2.5, 97.5])


mtest = split == "test"
main = ["multi-qa-MiniLM", "all-MiniLM", "bge-small", "e5-small"]
rows = []
for m in main:
    mean, lo, hi = boot_ci(nd[m][mtest])
    row = {"model": m, "n_queries": int(mtest.sum()), "ndcg@10": mean, "ci_lo": lo, "ci_hi": hi, "recall@10": runs[m]["recall"][mtest].mean()}
    if m != BASELINE:
        dm, dlo, dhi = boot_ci((nd[m] - nd[BASELINE])[mtest])
        row.update({"diff_vs_baseline": dm, "diff_ci_lo": dlo, "diff_ci_hi": dhi})
    rows.append(row)
pd.DataFrame(rows).round(4)

## Query groups

The primary grouping is KMeans (k = 50) on the query embeddings of the baseline model. Each group has approximately 65 queries. All group-level bootstrap and sign-flip procedures in the next sections use these groups.

In [ ]:
groups = KMeans(GROUP_K, n_init=5, random_state=SEED).fit_predict(runs[BASELINE]["Q"])
uniq, inv = np.unique(groups, return_inverse=True)
members = [np.where(inv == g)[0] for g in range(len(uniq))]
print("groups:", len(uniq), "| sizes: min", min(map(len, members)), "median", int(np.median([len(m) for m in members])), "max", max(map(len, members)))

## Paired comparison with the baseline

All models use their own default `max_seq_length`. For a contrast `d_i = nDCG_model(i) - nDCG_baseline(i)`:

- **CI (iid)** and **CI (cluster)**: percentile bootstrap over queries and over groups. The ratio of their widths is the design effect.
- **p (sign-flip, query)**: change the sign of each `d_i` at random (2,000 times), and compare the mean with the observed mean. This is valid if each `d_i` is symmetric about zero under the null.
- **p (sign-flip, group)**: change the sign of full groups. This keeps the dependence in each group. This is the primary p-value.
- **Holm** correction over the three pre-specified contrasts, on the group-level p-values.
- **dz** is the standardized paired effect (mean / SD of `d_i`).

**Why this test**
- **Estimand:** the mean paired difference in nDCG@10 between each model and the baseline, over all labelled NFCorpus queries.
- **Unit of independence:** the KMeans(50) topic group, not the query. Queries on one topic share relevant documents, so their differences are correlated. A resample or sign-flip of full groups keeps that correlation. Dependence can extend beyond k = 50, so the group CI is a lower limit on the uncertainty.
- **Assumptions:** the sign-flip test assumes that the summed difference of each group is symmetric about zero under "no difference". This is true if the two models are exchangeable in a group. The group bootstrap assumes that groups are independent draws.
- **Why not the usual alternative:** a paired t-test or a Wilcoxon test over queries treats queries as independent. The design effect (CI width ratio > 1) shows that they are not. A t-test also assumes approximately normal differences, and these differences are zero-inflated. Holm has the same error guarantee as Bonferroni and more power for three pre-specified contrasts.
- **Resolution / Monte Carlo error:** the lowest possible sign-flip p-value is 1/(B+1) = 0.0005. Thus, the lowest Holm value over three contrasts is 0.0015. This notebook reports values at this limit as `p < 0.002`.

In [ ]:
def boot_mean(d, B=B, labels_members=None, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(d)
    if labels_members is None:
        return np.array([d[rng.integers(0, n, n)].mean() for _ in range(B)])
    out = np.empty(B)
    for b in range(B):
        idx = np.concatenate([labels_members[g] for g in rng.integers(0, len(labels_members), len(labels_members))])
        out[b] = d[idx].mean()
    return out


def signflip_p(d, B=B, members_=None, seed=SEED):
    rng = np.random.default_rng(seed)
    obs = d.mean()
    if members_ is None:
        null = np.array([(d * rng.choice([-1.0, 1.0], len(d))).mean() for _ in range(B)])
    else:
        sums = np.array([d[m].sum() for m in members_])
        null = np.array([(sums * rng.choice([-1.0, 1.0], len(sums))).sum() / len(d) for _ in range(B)])
    return (1 + (np.abs(null) >= abs(obs)).sum()) / (1 + B)


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    out, run = np.empty(len(p)), 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, p[i] * (len(p) - rank)))
        out[i] = run
    return out


def contrast(a, b=BASELINE, mask=None):
    d = (nd[a] - nd[b]).to_numpy()
    mem = members
    if mask is not None:
        d = d[mask]
        remap = {q: i for i, q in enumerate(np.where(mask)[0])}
        mem = [m for m in (np.array([remap[i] for i in m if i in remap]) for m in members) if len(m)]
    bi, bc = boot_mean(d), boot_mean(d, labels_members=mem)
    ci_i, ci_c = np.percentile(bi, [2.5, 97.5]), np.percentile(bc, [2.5, 97.5])
    return {"n": len(d), "mean_diff": d.mean(), "sd_d": d.std(ddof=1), "dz": d.mean() / d.std(ddof=1),
            "ci_lo_iid": ci_i[0], "ci_hi_iid": ci_i[1], "ci_lo_cluster": ci_c[0], "ci_hi_cluster": ci_c[1],
            "design_effect": np.ptp(ci_c) / np.ptp(ci_i), "p_signflip_query": signflip_p(d),
            "p_signflip_group": signflip_p(d, members_=mem), "d": d, "mem": mem}


primary = ["all-MiniLM", "bge-small", "e5-small"]
store, rows = {}, []
for m in primary:
    store[m] = contrast(m)
    rows.append({"model": m, **{k: v for k, v in store[m].items() if k not in ("d", "mem")}})
pair = pd.DataFrame(rows)
pair["p_holm_group"] = holm(pair["p_signflip_group"])
pair.insert(1, "mean_nDCG@10", [nd[m].mean() for m in primary])
print("baseline mean nDCG@10:", round(float(nd[BASELINE].mean()), 4), "| queries:", len(nd))
pair.round(4)

## Stability of the topic grouping

KMeans makes the topic groups, so the groups depend on a random seed. The primary analysis uses seed 0. The CIs and p-values must not depend on this seed. This section calculates the group-level CI and sign-flip p-value again with five different KMeans seeds. The spread across seeds is the part of the uncertainty that comes from the grouping, not from the data.

In [ ]:
rows = []
for gs in range(5):
    lab_s = KMeans(GROUP_K, n_init=5, random_state=gs).fit_predict(runs[BASELINE]["Q"])
    uniq_s, inv_s = np.unique(lab_s, return_inverse=True)
    members_s = [np.where(inv_s == g)[0] for g in range(len(uniq_s))]
    for m in primary:
        d = (nd[m] - nd[BASELINE]).to_numpy()
        lo, hi = np.percentile(boot_mean(d, labels_members=members_s), [2.5, 97.5])
        rows.append({"grouping_seed": gs, "model": m, "mean_diff": d.mean(), "ci_lo": lo, "ci_hi": hi, "p_signflip_group": signflip_p(d, members_=members_s)})
gs_table = pd.DataFrame(rows)
gs_table.groupby("model").agg(mean_diff=("mean_diff", "first"), ci_lo_min=("ci_lo", "min"), ci_lo_max=("ci_lo", "max"),
                              ci_hi_min=("ci_hi", "min"), ci_hi_max=("ci_hi", "max"), p_max=("p_signflip_group", "max")).round(4)

## Does the test-split result generalize?

If the model effects are the same across splits, a test-only comparison agrees with the all-query comparison. This section gives the mean difference for each split with a cluster-bootstrap CI. It also gives a permutation test of heterogeneity across splits. The statistic is the range of the mean differences of the splits. The null shuffles the split labels **in each KMeans group** (2,000 times).

**Why this test**
- **Estimand:** how much the difference between a model and the baseline changes across train, validation, and test. The measure is the range of the means of the splits.
- **Unit of independence:** queries in a topic group are exchangeable for the split label. Full topics are not exchangeable.
- **Assumptions:** under "no heterogeneity", the split labels are exchangeable among queries of the same topic group.
- **Why not the usual alternative:** a shuffle of split labels across all queries mixes topics between splits. If the splits have different topic mixes, this adds variation to the null that the observed statistic does not have. It also ignores the dependence that the other sections include. A permutation in each group keeps the split composition of each group. Thus, the null changes only the property that the hypothesis is about. Cochran's Q or a chi-square test on the split estimates needs standard errors that are independent across splits. Here, the splits share topics and relevant documents, so they are not independent.
- **Resolution / Monte Carlo error:** 2,000 permutations. The lowest possible p-value is 1/2,001 ≈ 0.0005. The Monte Carlo SE of a p-value near 0.05 is approximately 0.005.

In [ ]:
def permute_within(labels_arr, members_, rng):
    out = labels_arr.copy()
    for m in members_:
        out[m] = rng.permutation(labels_arr[m])
    return out


rows = []
for m in primary:
    d = store[m]["d"]
    per = {}
    for s in SPLITS:
        c = contrast(m, mask=split == s)
        per[s] = c["mean_diff"]
        rows.append({"model": m, "split": s, "n": c["n"], "mean_diff": c["mean_diff"], "ci_lo_cluster": c["ci_lo_cluster"], "ci_hi_cluster": c["ci_hi_cluster"]})
    rng = np.random.default_rng(SEED)
    obs = np.ptp(list(per.values()))
    null = np.array([np.ptp([d[sh == s].mean() for s in SPLITS]) for sh in (permute_within(split, members, rng) for _ in range(2000))])
    rows.append({"model": m, "split": "heterogeneity (range of split means, within-group permutation)", "n": len(d), "mean_diff": obs,
                 "ci_lo_cluster": np.nan, "ci_hi_cluster": np.nan, "perm_p": (1 + (null >= obs).sum()) / 2001})
pd.DataFrame(rows).round(4)

## Equivalence (TOST)

A CI that includes zero is not evidence of no difference. The data supports the equivalence claim `|mean difference| < delta` when the 90% cluster-bootstrap CI is fully in `(-delta, +delta)`. This CI is equal to two one-sided 5% tests. The value `delta = 0.01` nDCG@10 was set in advance. A grid (0.005, 0.01, 0.02) shows how the result depends on delta. This section applies the test to these contrasts: the three model contrasts, e5-small against bge-small, and the default length against the other length for both MiniLM models.

**Why this test**
- **Estimand:** whether the mean paired difference is in ±delta. Delta is the largest difference that has no practical importance.
- **Unit of independence:** the KMeans(50) group, as in the main comparison.
- **Assumptions:** the 90% percentile interval has approximately correct coverage. Each one-sided 5% test rejects when the interval excludes the related limit.
- **Why not the usual alternative:** "the 95% CI includes zero" only fails to show a difference. It gives no information about equivalence. TOST is the standard test for "no meaningful difference". It uses a 90% interval because each of its two one-sided tests is at 5%.
- **Resolution / Monte Carlo error:** B = 2,000. The Monte Carlo error on a 5% endpoint is approximately 0.001. Thus, a result with an endpoint within 0.001 of ±delta is not stable.

In [ ]:
def tost(a, b):
    d = (nd[a] - nd[b]).to_numpy()
    lo90, hi90 = np.percentile(boot_mean(d, labels_members=members), [5, 95])
    row = {"contrast": f"{a} vs {b}", "mean_diff": d.mean(), "ci90_lo": lo90, "ci90_hi": hi90}
    for delta in (0.005, 0.01, 0.02):
        row[f"equivalent_d{delta}"] = bool(lo90 > -delta and hi90 < delta)
    return row


pairs = [("all-MiniLM", BASELINE), ("bge-small", BASELINE), ("e5-small", BASELINE), ("e5-small", "bge-small"),
         (BASELINE, ALT_NAME[BASELINE]), ("all-MiniLM", ALT_NAME["all-MiniLM"])]
pd.DataFrame([tost(a, b) for a, b in pairs]).round(4)

## Minimum detectable effect

This section uses `n` paired queries and the SD of the paired difference `sd_d`. A two-sided 5% test with 80% power detects `|d| >= (z_0.975 + z_0.80) * sd_d / sqrt(n_eff)`. With group dependence, the effective sample size is `n / design_effect^2`. The table uses the measured `sd_d` and design effect of each contrast. It also gives the same value for a design with only the 323 test queries.

**Why this calculation**
- **Estimand:** the smallest true mean difference that this design detects with 80% power. It shows whether a test-only comparison is sufficiently large. It does not show whether an effect is present.
- **Unit of independence:** groups, through `n_eff = n / design_effect^2`. Here, the design effect is a ratio of CI widths, so its square is the variance inflation.
- **Assumptions:** a normal approximation for the mean difference. This is reasonable at n ≥ 323. The calculation treats the measured `sd_d` as known.
- **Why not the usual alternative:** a simulated power calculation needs a generative model for the differences of each query. The analytic form is clear, and this notebook uses it only to compare designs.
- **Resolution:** a planning value with no Monte Carlo error. It includes the uncertainty in `sd_d` and in the design effect.

In [ ]:
z = stats.norm.ppf(0.975) + stats.norm.ppf(0.80)
rows = []
for m in primary:
    c = store[m]
    n_eff = c["n"] / c["design_effect"] ** 2
    rows.append({"model": m, "n": c["n"], "sd_d": c["sd_d"], "design_effect": c["design_effect"], "n_eff": n_eff,
                 "MDE_all_queries": z * c["sd_d"] / np.sqrt(n_eff), "MDE_323_queries_iid": z * c["sd_d"] / np.sqrt(323), "observed_diff": c["mean_diff"]})
pd.DataFrame(rows).round(4)

## Sensitivity: prefixes, sequence length, and a larger model

This section gives the mean nDCG@10 of each configuration. It also gives the paired difference from the natural reference of the configuration, with a cluster-bootstrap CI. These contrasts are **exploratory**, with Benjamini-Hochberg (BH) correction across the rows. A missing or incorrect prefix, or an unsuitable sequence length, can make a model look weaker for an incorrect reason. The last rows repeat the three primary contrasts with **all models forced to 512 tokens**. These rows show how the sequence length changes the primary contrasts.

**Why this test**
- **Estimand:** for each configuration, the mean paired difference from its natural reference. The reference is the same model with its default settings, or bge-small for bge-base.
- **Unit of independence:** the KMeans(50) group, as in the main comparison.
- **Assumptions:** the same as for the group sign-flip and the group bootstrap. BH assumes that the p-values are independent or positively dependent. This is plausible here, because the contrasts use the same queries.
- **Why not the usual alternative:** Holm controls the family-wise error rate, and this is too strict for an exploratory screen. BH controls the false discovery rate. This agrees with the goal: find the settings that need more examination.
- **Resolution / Monte Carlo error:** the lowest possible sign-flip p-value is 0.0005.

In [ ]:
refs = {"bge-small, no query prefix": "bge-small", "e5-small, no prefixes": "e5-small", "e5-small, query prefix only": "e5-small",
        ALT_NAME[BASELINE]: BASELINE, ALT_NAME["all-MiniLM"]: "all-MiniLM", "bge-base": "bge-small"}


def at512(m):
    """Return the configuration name of model m at 512 tokens. If the default is 512, return the default run."""
    if DEFAULT_SEQ[m] == 512:
        return m
    if m in ALT_NAME and ALT[m] == 512:
        return ALT_NAME[m]
    return None


# Forced-512 design: each primary model at 512 against the baseline at 512 (if both runs exist)
old_design = {f"{m} @512 vs {BASELINE} @512 (forced-512 design)": (at512(m), at512(BASELINE))
              for m in primary if at512(m) and at512(BASELINE)}

rows = []
for cfg, ref in list(refs.items()) + list(old_design.values()):
    c = contrast(cfg, b=ref)
    rows.append({"configuration": cfg, "reference": ref, "mean_nDCG@10": nd[cfg].mean(), "mean_diff": c["mean_diff"],
                 "ci_lo_cluster": c["ci_lo_cluster"], "ci_hi_cluster": c["ci_hi_cluster"], "p_signflip_group": c["p_signflip_group"]})
for row, name in zip(rows[len(refs):], old_design):
    row["configuration"] = name
sens = pd.DataFrame(rows)
p = sens["p_signflip_group"].to_numpy()
order = np.argsort(p)
adj = np.minimum.accumulate((p[order] * len(p) / (np.arange(len(p)) + 1))[::-1])[::-1]
sens["p_bh"] = np.empty(len(p))
sens.loc[order, "p_bh"] = np.minimum(adj, 1.0)
sens.round(4)

In [ ]:
order_ = main + ["bge-base"]
fig, ax = plt.subplots(figsize=(6, 3))
for i, m in enumerate(order_):
    d = nd[m].to_numpy()
    b = boot_mean(d, labels_members=members, B=500)
    ax.errorbar(i, d.mean(), yerr=[[d.mean() - np.percentile(b, 2.5)], [np.percentile(b, 97.5) - d.mean()]], fmt="o", capsize=3)
ax.set_xticks(range(len(order_)), order_, rotation=20, fontsize=8)
ax.set_ylabel("mean nDCG@10 (95% cluster CI)")
fig.tight_layout()
plt.show()

## Agreement between models

This section compares each pair of the four main models over all labelled queries. It gives two values: the Spearman correlation of nDCG@10 for each query, and the mean top-10 Jaccard overlap. A high nDCG correlation with a low top-10 overlap shows one thing. The models succeed and fail on the same queries, but they retrieve different documents.

In [ ]:
def pair_matrix(fn):
    return pd.DataFrame([[fn(a, b) for b in main] for a in main], index=main, columns=main)


corr = pair_matrix(lambda a, b: stats.spearmanr(runs[a]["ndcg"], runs[b]["ndcg"])[0])
jac = pair_matrix(lambda a, b: np.mean([len(set(x) & set(y)) / len(set(x) | set(y)) for x, y in zip(runs[a]["ids"], runs[b]["ids"])]))
print("per-query nDCG Spearman"); display(corr.round(2))
print("top-10 Jaccard"); jac.round(2)

In [ ]:
import datasets, matplotlib, sentence_transformers, sklearn, scipy, torch
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "matplotlib": matplotlib.__version__, "scipy": scipy.__version__,
       "sklearn": sklearn.__version__, "torch": torch.__version__, "sentence_transformers": sentence_transformers.__version__,
       "datasets": datasets.__version__, "seed": SEED, "device": "cpu", "group_k": GROUP_K, "bootstrap_B": B,
       "equivalence_margin": DELTA, "bge_base_revision": bge_base_rev})

## Findings
All numbers come from one full run (seed 0, NFCorpus, 3,237 queries). Each model uses its own default `max_seq_length`: multi-qa-MiniLM 512, all-MiniLM 256, bge-small 512, e5-small 512, bge-base 512.

- **Three models are better than the baseline on the pooled 3,237 queries** (baseline mean nDCG@10 0.296). bge-small +0.045 (cluster CI 0.038 to 0.052), e5-small +0.029 (0.023 to 0.036), all-MiniLM +0.023 (0.015 to 0.030). All three are significant after Holm (adjusted p < 0.002, the resolution limit).
- **The all-MiniLM gain depends on the sequence length.** It is +0.023 at its default of 256 tokens and +0.017 (0.009 to 0.024) when forced to 512. The other two models have a default of 512, so their values do not change.
- **The ordering is only partly supported.** bge-small is better than e5-small (difference 0.016, 90% CI 0.011 to 0.020, from the TOST table; not a pre-specified contrast). e5-small and all-MiniLM are 0.0067 apart, and this notebook does not test this contrast. Thus, the data does not support an order between e5-small and all-MiniLM, or between bge-small and all-MiniLM.
- **The result is for the pooled queries. It is not shown for each split.** The heterogeneity p-values are 0.45, 0.43, and 0.039 (e5-small). The e5-small value is below 0.05 before Holm, and 0.117 after Holm over the three models. On the 323 test queries alone, the all-MiniLM CI is +0.002 to +0.035, which is near zero.
- **The effects are small.** The standardized paired effects are dz 0.16 to 0.29. This is 8% to 15% of the nDCG of the baseline. No model contrast with the baseline is equivalent within 0.02. The 90% CI of each contrast is fully above 0.01.
- **Test-only comparisons are underpowered.** The minimum detectable effect is approximately 0.009 to 0.011 on all queries, and approximately 0.022 to 0.025 on 323 queries. The test-only differences (0.019 to 0.046) are at or a small distance above this value.
- **Prefixes have a small effect.** Without its query instruction, bge-small loses 0.004. Without its prefixes, e5-small loses 0.005. Both are significant after BH (p = 0.010). With only its query prefix, e5-small loses 0.001 (not significant).
- **Sequence length has an effect for all-MiniLM, but not for the baseline.** For the baseline, 256 against its default 512 gives +0.0001 (equivalent within 0.005). For all-MiniLM, the nDCG@10 is 0.3186 at its default 256 and 0.3126 at 512 (-0.006 at 512, BH p = 0.002). This difference is equivalent within 0.01 but not within 0.005. Thus, a difference can be real and also smaller than the margin.
- **A larger model gives a larger gain.** bge-base is +0.024 over bge-small (exploratory).
- **The models agree on which queries are difficult, but not on which documents to return.** The Spearman correlation of nDCG for each query is 0.86 to 0.91. The mean top-10 Jaccard overlap is 0.28 to 0.33.

### What this supports
This result is for NFCorpus, all 3,237 labelled queries, and each model at its shipped settings. Here, the choice among these small encoders changes mean nDCG@10 by a few hundredths. All three models are better than the baseline, and bge-small is better than e5-small. These results are for the pooled queries.

### What this does not support
- A full order of the models. This notebook does not test e5-small or bge-small against all-MiniLM.
- That the gains are the same in each split. The e5-small heterogeneity p-value is 0.039 before Holm, and the test-only all-MiniLM CI is near zero.
- One value for the all-MiniLM gain. It is +0.017 to +0.023, and it depends on `max_seq_length`.
- A decision about chunking (issue #5). This notebook does not use chunking.
- That the order, or the size of the gaps, is the same on other datasets or domains. SciFact and FiQA are held out.
- That the result is the same for larger or differently trained encoders, or a reason for the differences between models.
- That the differences are important for the quality of the final answers.
- That the p-values are exact. The cluster CIs use KMeans(50) groups, and dependence can extend beyond these groups. Thus, the intervals are lower limits on the uncertainty.
- That absolute nDCG measures quality. Unjudged documents count as non-relevant. This penalizes models that are not similar to the systems that made the judgment pool.
- That these numbers are equal to the clean baseline of the benchmark runner in the fourth decimal. The runner breaks ties with FAISS, not with a stable sort.